# Measuring Stock Seasonality

## Proposed pipeline:

1. <b>Augmented Dickey-Fuller test on raw closing prices</b>:

   Check if series if non-stationary (probably will be). If not, skip to step 3.

2. <b>ADF on Intraday_Action_Pct</b>:

   If stationary, proceed.

3. <b>STL decomposition</b>:

   Confirm seasonality actually exists and identify periods (weekly, monthly, quarterly, yearly).

4. <b>SARIMA/SARIMAX</b>:

   If there is correlation between tickers, run SARIMAX with a competing ticker Intraday_Action_Pct as an exogenous variable. Otherwise, use SARIMA.


In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

TICKERS = ["SONY", "EA", "NTES", "NFLX", "DIS", "CMSA", "SPOT", "LYV", "WMG"]
df = pd.read_parquet("../stock_data.parquet")
ticker_data = {ticker: df[df["Ticker"] == ticker].copy() for ticker in TICKERS}

## Augmented Dickey-Fuller


In [ ]:
from statsmodels.tsa.stattools import adfuller


def adf_test(ticker_data, series_name):
    results = []
    for name, df in ticker_data.items():
        series = df[series_name].dropna()
        adf_stat, p_value, _, _, critical_values, _ = adfuller(series)
        results.append(
            {
                "Ticker": name,
                "ADF Statistic": round(adf_stat, 4),
                "p-value": round(p_value, 4),
                "Critical Value (1%)": round(critical_values["1%"], 4),
                "Critical Value (5%)": round(critical_values["5%"], 4),
                "Stationary (5%)": p_value < 0.05,
            }
        )

    return pd.DataFrame(results).set_index("Ticker")


def visualize_adf_results(results_df, series_name):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(
        f"ADF Test Results — {series_name} by Ticker",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )

    colors = ["#2ecc71" if s else "#e74c3c" for s in results_df["Stationary (5%)"]]

    # ADF Statistic vs Critical Values
    ax1 = axes[0]
    x = np.arange(len(results_df))
    bars = ax1.bar(
        x,
        results_df["ADF Statistic"],
        color=colors,
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )

    ax1.axhline(
        results_df["Critical Value (5%)"].iloc[0],
        color="orange",
        linestyle="--",
        linewidth=1.5,
        label="Critical Value (5%)",
        zorder=4,
    )
    ax1.axhline(
        results_df["Critical Value (1%)"].iloc[0],
        color="red",
        linestyle="--",
        linewidth=1.5,
        label="Critical Value (1%)",
        zorder=4,
    )

    ax1.set_xticks(x)
    ax1.set_xticklabels(results_df.index, rotation=30, ha="right")
    ax1.set_ylabel("ADF Statistic")
    ax1.set_title("ADF Statistic vs Critical Values\n(More negative = more stationary)")
    ax1.legend()
    ax1.grid(axis="y", alpha=0.3, zorder=0)
    ax1.set_axisbelow(True)

    for bar, val in zip(bars, results_df["ADF Statistic"]):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() - 0.3,
            f"{val:.2f}",
            ha="center",
            va="top",
            fontsize=9,
            color="white",
            fontweight="bold",
        )

    # p-values
    ax2 = axes[1]
    bars2 = ax2.bar(
        x,
        results_df["p-value"],
        color=colors,
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )

    ax2.axhline(
        0.05, color="orange", linestyle="--", linewidth=1.5, label="α = 0.05", zorder=4
    )
    ax2.axhline(
        0.01, color="red", linestyle="--", linewidth=1.5, label="α = 0.01", zorder=4
    )

    ax2.set_xticks(x)
    ax2.set_xticklabels(results_df.index, rotation=30, ha="right")
    ax2.set_ylabel("p-value")
    ax2.set_title("p-values by Ticker\n(Below threshold = reject unit root)")
    ax2.legend()
    ax2.grid(axis="y", alpha=0.3, zorder=0)
    ax2.set_axisbelow(True)

    for bar, val in zip(bars2, results_df["p-value"]):
        ypos = bar.get_height() + 0.001
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            ypos,
            f"{val:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )

    # stationary status
    green_patch = mpatches.Patch(color="#2ecc71", label="Stationary (p < 0.05)")
    red_patch = mpatches.Patch(color="#e74c3c", label="Non-stationary (p ≥ 0.05)")
    fig.legend(
        handles=[green_patch, red_patch],
        loc="lower center",
        ncol=2,
        bbox_to_anchor=(0.5, -0.08),
        frameon=False,
    )

    plt.tight_layout()
    plt.savefig("adf_results.png", dpi=150, bbox_inches="tight")
    plt.show()

    # summary
    print(results_df.to_string())


close_adf_results = adf_test(ticker_data, "Close")
intraday_adf_results = adf_test(ticker_data, "Intraday_Action_Pct")

visualize_adf_results(close_adf_results, "Close")
visualize_adf_results(intraday_adf_results, "Intraday_Action_Pct")